In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ MSE = M = \frac{1}{N} \sum_{i=1}^N (y_i - p_i)^2 $$

$$ \frac{\partial M}{\partial p} = \frac{2}{N} (p - y) $$

$$ \frac{\partial M}{\partial y} = -\frac{2}{N} (p - y) $$

$$ \diamond \diamond \diamond $$

$$ \text{Let } p = xW+b $$


In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore


def mse(p: tr.Tensor, y: tr.Tensor) -> tr.Tensor:
    """Computes the mean squared error between predictions `p` and targets `y`."""
  
    L = p - y
    L = (L * L).mean()
    return L


class MeanSquaredErrorFunction(autograd.Function):
    """Custom autograd function for mean squared error."""

    @staticmethod
    def forward(ctx, p: tr.Tensor, y: tr.Tensor) -> tr.Tensor:
        ctx.save_for_backward(p, y)
        L = mse(p, y)
        return L
    
    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, tr.Tensor]:
        (p, y) = ctx.saved_tensors
        S = p.shape[0]
        FO = p.shape[1]

        dM_dp = 2 * (p - y)

        # Average over all elements
        N = S * FO
        dM_dp = dM_dp / N

        # For this example dF_df == 1
        dM_dp = dM_dp * dF_df
        assert dM_dp.shape == (S, FO)

        dM_dy = -dM_dp
        assert dM_dy.shape == (S, FO)
        
        return (dM_dp, dM_dy)
    

class MeanSquaredError(nn.Module):
    """Mean squared error loss module."""

    def __init__(self):
        super().__init__()
    
    def forward(self, p: tr.Tensor, y: tr.Tensor) -> tr.Tensor:
        L = MeanSquaredErrorFunction.apply(p, y)
        return L
    

def test_basic():
  p = tr.tensor([0.2, -1.0, 3.0, 2.5], dtype=tr.float32)
  y = tr.tensor([0.0, -2.0, 1.0, 2.0], dtype=tr.float32)

  actual = mse(p, y)
  expected = tr.nn.MSELoss()(p, y)

  assert actual == approx(expected)


def test_gradcheck(samples) -> None:
    def fn(p, y):
        return MeanSquaredErrorFunction.apply(p, y)

    p = tr.randn((samples, 1), dtype=tr.float64, requires_grad=True)
    y = tr.randn((samples, 1), dtype=tr.float64, requires_grad=True)

    assert autograd.gradcheck(fn, (p, y), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    p = tr.randn(samples, dtype=tr.float32, requires_grad=True)
    y = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    mse_custom = MeanSquaredErrorFunction.apply(p, y)
    mse_builtin = tr.nn.MSELoss()(p, y)

    assert mse_custom == approx(mse_builtin)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)
